# TCGA-BRCA Clinical Shortlist Review

This notebook is review-only. It loads the latest saved clinical shortlist outputs from disk, checks shortlist-level summaries, and writes review tables for human source audit.

Important reminders:

- this notebook remains part of source audit
- this notebook does not freeze the cohort
- this notebook does not freeze the endpoint
- this notebook does not parse raw files or rebuild the core field-audit outputs


## Load the latest saved clinical shortlist run

This section confirms that the stable latest-pointer exists and points to a completed clinical shortlist run.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root / '01-data' / 'audit' / 'tcga-brca' / 'variables' / 'tcga_brca_clinical_shortlist_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest clinical shortlist pointer not found: {latest_pointer_path}. Run the shortlist script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
shortlist_path = repo_root / latest_pointer['clinical_shortlist_tsv']
shortlist_summary_path = repo_root / latest_pointer['clinical_shortlist_summary_tsv']
shortlist_by_table_path = repo_root / latest_pointer['clinical_shortlist_by_table_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_pointer]))


,updated_at_utc,shortlist_run_id,core_audit_run_id,parse_run_id,source_run_id,shortlist_profile,shortlist_run_directory,clinical_shortlist_tsv,clinical_shortlist_summary_tsv,clinical_shortlist_by_table_tsv,run_log_json,core_audit_latest_json,field_count
0,2026-04-12T03:36:02Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,balanced,01-data/audit/tcga-brca/variables/clinical_sho...,01-data/audit/tcga-brca/variables/clinical_sho...,01-data/audit/tcga-brca/variables/clinical_sho...,01-data/audit/tcga-brca/variables/clinical_sho...,01-data/audit/tcga-brca/variables/clinical_sho...,01-data/audit/tcga-brca/variables/tcga_brca_cl...,171


## Load saved shortlist artifacts

This section reads the saved shortlist TSVs and run log from disk only.


In [2]:
shortlist_df = pd.read_csv(shortlist_path, sep='\t')
shortlist_summary_df = pd.read_csv(shortlist_summary_path, sep='\t')
shortlist_by_table_df = pd.read_csv(shortlist_by_table_path, sep='\t')
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

bool_columns = [
    'support_high_completeness',
    'support_high_missingness',
    'support_likely_high_value',
    'support_likely_weak',
]
for column_name in bool_columns:
    shortlist_df[column_name] = shortlist_df[column_name].astype(str).str.lower().map({'true': True, 'false': False})

bucket_order = [
    'usable_baseline',
    'usable_treatment_proxy',
    'usable_endpoint_candidate',
    'weak_or_unusable',
    'unclear_manual_review',
]
priority_order = ['high', 'medium', 'low']
shortlist_df['shortlist_bucket'] = pd.Categorical(shortlist_df['shortlist_bucket'], categories=bucket_order, ordered=True)
shortlist_df['manual_review_priority'] = pd.Categorical(
    shortlist_df['manual_review_priority'],
    categories=priority_order,
    ordered=True,
)
shortlist_summary_df['shortlist_bucket'] = pd.Categorical(
    shortlist_summary_df['shortlist_bucket'],
    categories=bucket_order,
    ordered=True,
)
shortlist_by_table_df['shortlist_bucket'] = pd.Categorical(
    shortlist_by_table_df['shortlist_bucket'],
    categories=bucket_order,
    ordered=True,
)

print(f'Shortlist TSV: {shortlist_path}')
print(f'Shortlist summary TSV: {shortlist_summary_path}')
print(f'Shortlist by-table TSV: {shortlist_by_table_path}')
print(f'Run log: {run_log_path}')
print(f"Shortlist run ID: {latest_pointer['shortlist_run_id']}")
print(f"Core audit run ID: {latest_pointer['core_audit_run_id']}")
display(pd.DataFrame([run_log['validation']]))
display(shortlist_df.head(10))


Shortlist TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\clinical_shortlist_runs\20260412T033602Z\clinical_shortlist.tsv
Shortlist summary TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\clinical_shortlist_runs\20260412T033602Z\clinical_shortlist_summary.tsv
Shortlist by-table TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\clinical_shortlist_runs\20260412T033602Z\clinical_shortlist_by_table.tsv
Run log: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\clinical_shortlist_runs\20260412T033602Z\run_log.json
Shortlist run ID: 20260412T033602Z
Core audit run ID: 20260412T023839Z


,passed,input_audit_tables_found,input_group_summary_found,input_missingness_summary_found,input_review_tables_found,output_row_count_matches_core_audit,every_field_has_exactly_one_shortlist_bucket,summary_bucket_counts_match_total,by_table_bucket_counts_match_table_totals,no_prior_run_overwrite
0,True,True,True,True,True,True,True,True,True,True


,shortlist_run_id,core_audit_run_id,parse_run_id,source_run_id,table_name,field_name,source_position,alternate_column_name,probable_field_group,missing_like_fraction,...,distinct_non_missing_count,example_values_small_sample,support_high_completeness,support_high_missingness,support_likely_high_value,support_likely_weak,shortlist_bucket,shortlist_rule,manual_review_priority,notes_placeholder
0,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,bcr_patient_uuid,1,bcr_patient_uuid,identifier / admin,0.000000,...,780,"[""C07B122E-AC50-4DB2-ADD2-5617A5D0E976"", ""9435...",True,False,False,False,unclear_manual_review,manual_identifier_admin,high,[fill in during shortlist review]
1,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,bcr_patient_barcode,2,bcr_patient_barcode,identifier / admin,0.000000,...,780,"[""TCGA-GM-A2DA"", ""TCGA-AO-A03N"", ""TCGA-A2-A0EW...",True,False,False,False,unclear_manual_review,manual_identifier_admin,high,[fill in during shortlist review]
2,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,bcr_drug_barcode,3,bcr_drug_barcode,identifier / admin,0.000000,...,2406,"[""TCGA-3C-AAAU-D60350"", ""TCGA-3C-AALI-D62900"",...",True,False,False,False,unclear_manual_review,manual_identifier_admin,high,[fill in during shortlist review]
3,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,bcr_drug_uuid,4,bcr_drug_uuid,identifier / admin,0.000000,...,2406,"[""00300D73-B562-4EE9-A8D4-27E5B97AE501"", ""0051...",True,False,False,False,unclear_manual_review,manual_identifier_admin,high,[fill in during shortlist review]
4,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,form_completion_date,5,form_completion_date,identifier / admin,0.000000,...,345,"[""2012-12-6"", ""2011-1-10"", ""2010-9-19"", ""2013-...",True,False,False,False,unclear_manual_review,manual_identifier_admin,high,[fill in during shortlist review]
5,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_therapy_drug_name,6,drug_name,treatment / drug,0.006650,...,203,"[""Cytoxan"", ""Tamoxifen"", ""Cyclophosphamide"", ""...",True,False,True,False,usable_treatment_proxy,treatment_proxy_direct_treatment_table,medium,[fill in during shortlist review]
6,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,clinical_trial_drug_classification,7,clinical_trail_drug_classification,stage,0.998753,...,3,"[""Antimetabolite"", ""Biological Therapy/Monoclo...",False,True,False,True,weak_or_unusable,weak_extreme_missingness,low,[fill in during shortlist review]
7,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_therapy_type,8,therapy_type,treatment / drug,0.002909,...,8,"[""Chemotherapy"", ""Hormone Therapy"", ""Immunothe...",True,False,True,False,usable_treatment_proxy,treatment_proxy_direct_treatment_table,medium,[fill in during shortlist review]
8,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_tx_started_days_to,9,days_to_drug_therapy_start,treatment / drug,0.049044,...,486,"[""50"", ""61"", ""31"", ""62"", ""63""]",True,False,True,False,usable_treatment_proxy,treatment_proxy_direct_treatment_table,medium,[fill in during shortlist review]
9,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,pharmaceutical_tx_ongoing_indicator,10,therapy_ongoing,treatment / drug,0.006234,...,2,"[""NO"", ""YES""]",True,False,True,False,usable_treatment_proxy,treatment_proxy_direct_treatment_table,medium,[fill in during shortlist review]


## Save shortlist bucket counts overall and by table


In [3]:
bucket_counts_df = shortlist_summary_df.sort_values('shortlist_bucket').reset_index(drop=True)
bucket_counts_path = results_root / '25_clinical_shortlist_bucket_counts.tsv'
bucket_counts_df.to_csv(bucket_counts_path, sep='\t', index=False)

bucket_counts_by_table_df = shortlist_by_table_df.sort_values(['table_name', 'shortlist_bucket']).reset_index(drop=True)
bucket_counts_by_table_path = results_root / '26_clinical_shortlist_bucket_counts_by_table.tsv'
bucket_counts_by_table_df.to_csv(bucket_counts_by_table_path, sep='\t', index=False)

print(f'Saved: {bucket_counts_path}')
print(f'Saved: {bucket_counts_by_table_path}')
display(bucket_counts_df)
display(bucket_counts_by_table_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\25_clinical_shortlist_bucket_counts.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\26_clinical_shortlist_bucket_counts_by_table.tsv


,shortlist_run_id,core_audit_run_id,parse_run_id,source_run_id,shortlist_bucket,field_count,field_fraction,high_priority_count,medium_priority_count,low_priority_count,field_names_json
0,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,usable_baseline,27,0.157895,0,6,21,"[""birth_days_to"", ""gender"", ""menopause_status""..."
1,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,usable_treatment_proxy,15,0.087719,0,15,0,"[""pharmaceutical_therapy_drug_name"", ""pharmace..."
2,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,usable_endpoint_candidate,8,0.046784,8,0,0,"[""followup_lost_to"", ""tumor_status"", ""vital_st..."
3,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,weak_or_unusable,62,0.362573,0,0,62,"[""clinical_trial_drug_classification"", ""days_t..."
4,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,unclear_manual_review,59,0.345029,59,0,0,"[""bcr_patient_uuid"", ""bcr_patient_barcode"", ""b..."


,shortlist_run_id,core_audit_run_id,parse_run_id,source_run_id,table_name,shortlist_bucket,field_count,field_fraction_of_table,high_priority_count,medium_priority_count,low_priority_count,field_names_json
0,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,usable_baseline,0,0.000000,0,0,0,[]
1,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,usable_treatment_proxy,5,0.178571,0,5,0,"[""pharmaceutical_therapy_drug_name"", ""pharmace..."
2,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,usable_endpoint_candidate,0,0.000000,0,0,0,[]
3,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,weak_or_unusable,8,0.285714,0,0,8,"[""clinical_trial_drug_classification"", ""days_t..."
4,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_drug,unclear_manual_review,15,0.535714,15,0,0,"[""bcr_patient_uuid"", ""bcr_patient_barcode"", ""b..."
5,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,usable_baseline,0,0.000000,0,0,0,[]
6,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,usable_treatment_proxy,2,0.153846,0,2,0,"[""radiation_treatment_adjuvant"", ""pharmaceutic..."
7,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,usable_endpoint_candidate,5,0.384615,5,0,0,"[""followup_lost_to"", ""tumor_status"", ""vital_st..."
8,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,weak_or_unusable,1,0.076923,0,0,1,"[""death_days_to""]"
9,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,clinical_follow_up_v4_0,unclear_manual_review,5,0.384615,5,0,0,"[""bcr_patient_uuid"", ""bcr_patient_barcode"", ""b..."


## Save bucket-specific review tables

These remain shortlist and source-audit review queues only.


In [4]:
review_columns = [
    'table_name',
    'field_name',
    'source_position',
    'alternate_column_name',
    'probable_field_group',
    'missing_like_fraction',
    'non_missing_count',
    'distinct_non_missing_count',
    'support_high_completeness',
    'support_high_missingness',
    'support_likely_high_value',
    'support_likely_weak',
    'shortlist_rule',
    'manual_review_priority',
    'example_values_small_sample',
    'notes_placeholder',
]


In [5]:
usable_baseline_df = (
    shortlist_df.loc[shortlist_df['shortlist_bucket'] == 'usable_baseline', review_columns]
    .sort_values(
        ['manual_review_priority', 'missing_like_fraction', 'non_missing_count', 'table_name', 'source_position'],
        ascending=[True, True, False, True, True],
    )
    .reset_index(drop=True)
)
usable_baseline_path = results_root / '27_clinical_shortlist_usable_baseline_fields.tsv'
usable_baseline_df.to_csv(usable_baseline_path, sep='\t', index=False)

print(f'Saved: {usable_baseline_path}')
display(usable_baseline_df.head(25))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\27_clinical_shortlist_usable_baseline_fields.tsv


,table_name,field_name,source_position,alternate_column_name,probable_field_group,missing_like_fraction,non_missing_count,distinct_non_missing_count,support_high_completeness,support_high_missingness,support_likely_high_value,support_likely_weak,shortlist_rule,manual_review_priority,example_values_small_sample,notes_placeholder
0,clinical_patient,lymph_nodes_examined_count,34,lymph_node_examined_count,diagnosis / pathology,0.114859,971,42,False,False,True,False,baseline_candidate_structured_core_field,medium,"[""2"", ""1"", ""3"", ""4"", ""10""]",[fill in during shortlist review]
1,clinical_patient,ajcc_staging_edition,37,system_version,stage,0.128532,956,5,False,False,True,False,baseline_candidate_structured_core_field,medium,"[""6th"", ""7th"", ""5th"", ""4th"", ""3rd""]",[fill in during shortlist review]
2,clinical_patient,lymph_nodes_examined_he_count,35,number_of_lymphnodes_positive_by_he,diagnosis / pathology,0.153145,929,31,False,False,True,False,baseline_candidate_structured_core_field,medium,"[""0"", ""1"", ""2"", ""3"", ""4""]",[fill in during shortlist review]
3,clinical_patient,ethnicity,10,ethnicity,demographics,0.158614,923,2,False,False,True,False,baseline_candidate_structured_core_field,medium,"[""NOT HISPANIC OR LATINO"", ""HISPANIC OR LATINO""]",[fill in during shortlist review]
4,clinical_patient,her2_status_by_ihc,56,lab_proc_her2_neu_immunohistochemistry_recepto...,receptor / biomarker,0.162261,919,4,False,False,True,False,baseline_candidate_structured_core_field,medium,"[""Negative"", ""Equivocal"", ""Positive"", ""Indeter...",[fill in during shortlist review]
5,clinical_patient,axillary_staging_method,30,axillary_lymph_node_stage_method_type,diagnosis / pathology,0.196901,881,5,False,False,True,False,baseline_candidate_structured_core_field,medium,"[""Axillary lymph node dissection alone"", ""Sent...",[fill in during shortlist review]
6,clinical_patient,gender,7,gender,demographics,0.000000,1097,2,True,False,True,False,baseline_candidate_structured_core_field,low,"[""FEMALE"", ""MALE""]",[fill in during shortlist review]
7,clinical_patient,age_at_diagnosis,21,age_at_initial_pathologic_diagnosis,demographics,0.000000,1097,65,True,False,True,False,baseline_candidate_structured_core_field,low,"[""62"", ""63"", ""61"", ""50"", ""54""]",[fill in during shortlist review]
8,clinical_patient,ajcc_tumor_pathologic_pt,38,pathologic_T,stage,0.000000,1097,13,True,False,True,False,baseline_candidate_structured_core_field,low,"[""T2"", ""T1c"", ""T3"", ""T1"", ""T4b""]",[fill in during shortlist review]
9,clinical_patient,ajcc_nodes_pathologic_pn,39,pathologic_N,stage,0.000000,1097,16,True,False,True,False,baseline_candidate_structured_core_field,low,"[""N0"", ""N1a"", ""N0 (i-)"", ""N1"", ""N2a""]",[fill in during shortlist review]


In [6]:
usable_treatment_proxy_df = (
    shortlist_df.loc[shortlist_df['shortlist_bucket'] == 'usable_treatment_proxy', review_columns]
    .sort_values(
        ['manual_review_priority', 'missing_like_fraction', 'non_missing_count', 'table_name', 'source_position'],
        ascending=[True, True, False, True, True],
    )
    .reset_index(drop=True)
)
usable_treatment_proxy_path = results_root / '28_clinical_shortlist_usable_treatment_proxy_fields.tsv'
usable_treatment_proxy_df.to_csv(usable_treatment_proxy_path, sep='\t', index=False)

print(f'Saved: {usable_treatment_proxy_path}')
display(usable_treatment_proxy_df.head(25))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\28_clinical_shortlist_usable_treatment_proxy_fields.tsv


,table_name,field_name,source_position,alternate_column_name,probable_field_group,missing_like_fraction,non_missing_count,distinct_non_missing_count,support_high_completeness,support_high_missingness,support_likely_high_value,support_likely_weak,shortlist_rule,manual_review_priority,example_values_small_sample,notes_placeholder
0,clinical_patient,history_neoadjuvant_treatment,12,history_of_neoadjuvant_treatment,treatment / drug,0.001823,1095,2,True,False,True,False,treatment_proxy_treatment_signal,medium,"[""No"", ""Yes""]",[fill in during shortlist review]
1,clinical_drug,pharmaceutical_therapy_type,8,therapy_type,treatment / drug,0.002909,2399,8,True,False,True,False,treatment_proxy_direct_treatment_table,medium,"[""Chemotherapy"", ""Hormone Therapy"", ""Immunothe...",[fill in during shortlist review]
2,clinical_drug,pharmaceutical_tx_ongoing_indicator,10,therapy_ongoing,treatment / drug,0.006234,2391,2,True,False,True,False,treatment_proxy_direct_treatment_table,medium,"[""NO"", ""YES""]",[fill in during shortlist review]
3,clinical_drug,pharmaceutical_therapy_drug_name,6,drug_name,treatment / drug,0.006650,2390,203,True,False,True,False,treatment_proxy_direct_treatment_table,medium,"[""Cytoxan"", ""Tamoxifen"", ""Cyclophosphamide"", ""...",[fill in during shortlist review]
4,clinical_radiation,radiation_therapy_ongoing_indicator,12,radiation_treatment_ongoing,radiation,0.009709,612,2,True,False,True,False,treatment_proxy_direct_treatment_table,medium,"[""NO"", ""YES""]",[fill in during shortlist review]
5,clinical_radiation,radiation_therapy_type,6,radiation_type,radiation,0.030744,599,5,True,False,True,False,treatment_proxy_direct_treatment_table,medium,"[""External"", ""EXTERNAL BEAM"", ""OTHER"", ""IMPLAN...",[fill in during shortlist review]
6,clinical_radiation,radiation_therapy_site,7,anatomic_treatment_site,radiation,0.032362,598,5,True,False,True,False,treatment_proxy_direct_treatment_table,medium,"[""Primary Tumor Field"", ""Regional site"", ""Dist...",[fill in during shortlist review]
7,clinical_follow_up_v4_0,radiation_treatment_adjuvant,7,radiation_therapy,radiation,0.033520,692,2,True,False,True,False,treatment_proxy_treatment_signal,medium,"[""YES"", ""NO""]",[fill in during shortlist review]
8,clinical_radiation,radiation_therapy_ended_days_to,13,days_to_radiation_therapy_end,radiation,0.033981,597,275,True,False,True,False,treatment_proxy_direct_treatment_table,medium,"[""212"", ""246"", ""263"", ""275"", ""184""]",[fill in during shortlist review]
9,clinical_radiation,radiation_therapy_started_days_to,11,days_to_radiation_therapy_start,radiation,0.038835,594,267,True,False,True,False,treatment_proxy_direct_treatment_table,medium,"[""153"", ""181"", ""211"", ""218"", ""188""]",[fill in during shortlist review]


In [7]:
usable_endpoint_candidate_df = (
    shortlist_df.loc[shortlist_df['shortlist_bucket'] == 'usable_endpoint_candidate', review_columns]
    .sort_values(
        ['manual_review_priority', 'missing_like_fraction', 'non_missing_count', 'table_name', 'source_position'],
        ascending=[True, True, False, True, True],
    )
    .reset_index(drop=True)
)
usable_endpoint_candidate_path = results_root / '29_clinical_shortlist_usable_endpoint_candidate_fields.tsv'
usable_endpoint_candidate_df.to_csv(usable_endpoint_candidate_path, sep='\t', index=False)

print(f'Saved: {usable_endpoint_candidate_path}')
display(usable_endpoint_candidate_df.head(25))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\29_clinical_shortlist_usable_endpoint_candidate_fields.tsv


,table_name,field_name,source_position,alternate_column_name,probable_field_group,missing_like_fraction,non_missing_count,distinct_non_missing_count,support_high_completeness,support_high_missingness,support_likely_high_value,support_likely_weak,shortlist_rule,manual_review_priority,example_values_small_sample,notes_placeholder
0,clinical_patient,vital_status,14,vital_status,follow-up / outcome-like,0.000000,1097,2,True,False,True,False,endpoint_candidate_time_or_status_signal,high,"[""Alive"", ""Dead""]",[fill in during shortlist review]
1,clinical_follow_up_v4_0,vital_status,10,vital_status,follow-up / outcome-like,0.013966,706,2,True,False,True,False,endpoint_candidate_time_or_status_signal,high,"[""Alive"", ""Dead""]",[fill in during shortlist review]
2,clinical_follow_up_v4_0,followup_lost_to,6,lost_follow_up,follow-up / outcome-like,0.036313,690,2,True,False,True,False,endpoint_candidate_time_or_status_signal,high,"[""NO"", ""YES""]",[fill in during shortlist review]
3,clinical_follow_up_v4_0,tumor_status,9,person_neoplasm_cancer_status,follow-up / outcome-like,0.054469,677,2,True,False,True,False,endpoint_candidate_time_or_status_signal,high,"[""TUMOR FREE"", ""WITH TUMOR""]",[fill in during shortlist review]
4,clinical_follow_up_v4_0,last_contact_days_to,11,days_to_last_followup,follow-up / outcome-like,0.086592,654,551,True,False,True,False,endpoint_candidate_time_or_status_signal,high,"[""10"", ""0"", ""375"", ""396"", ""304""]",[fill in during shortlist review]
5,clinical_patient,last_contact_days_to,15,days_to_last_followup,follow-up / outcome-like,0.094804,993,649,True,False,True,False,endpoint_candidate_time_or_status_signal,high,"[""0"", ""10"", ""7"", ""365"", ""30""]",[fill in during shortlist review]
6,clinical_follow_up_v4_0,new_tumor_event_dx_indicator,13,new_tumor_event_after_initial_treatment,follow-up / outcome-like,0.096369,647,2,True,False,True,False,endpoint_candidate_time_or_status_signal,high,"[""NO"", ""YES""]",[fill in during shortlist review]
7,clinical_patient,tumor_status,13,person_neoplasm_cancer_status,follow-up / outcome-like,0.113947,972,2,False,False,True,False,endpoint_candidate_time_or_status_signal,high,"[""TUMOR FREE"", ""WITH TUMOR""]",[fill in during shortlist review]


In [8]:
weak_or_unusable_df = (
    shortlist_df.loc[shortlist_df['shortlist_bucket'] == 'weak_or_unusable', review_columns]
    .sort_values(
        ['missing_like_fraction', 'non_missing_count', 'distinct_non_missing_count', 'table_name', 'source_position'],
        ascending=[False, True, True, True, True],
    )
    .reset_index(drop=True)
)
weak_or_unusable_path = results_root / '30_clinical_shortlist_weak_or_unusable_fields.tsv'
weak_or_unusable_df.to_csv(weak_or_unusable_path, sep='\t', index=False)

print(f'Saved: {weak_or_unusable_path}')
display(weak_or_unusable_df.head(25))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\30_clinical_shortlist_weak_or_unusable_fields.tsv


,table_name,field_name,source_position,alternate_column_name,probable_field_group,missing_like_fraction,non_missing_count,distinct_non_missing_count,support_high_completeness,support_high_missingness,support_likely_high_value,support_likely_weak,shortlist_rule,manual_review_priority,example_values_small_sample,notes_placeholder
0,clinical_drug,days_to_stem_cell_transplantation,13,days_to_stem_cell_transplantation,treatment / drug,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]
1,clinical_drug,pharm_regimen,14,pharm_regimen,treatment / drug,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]
2,clinical_drug,pharm_regimen_other,15,pharm_regimen_other,treatment / drug,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]
3,clinical_drug,stem_cell_transplantation,23,stem_cell_transplantation,treatment / drug,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]
4,clinical_drug,stem_cell_transplantation_type,24,stem_cell_transplantation_type,treatment / drug,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]
5,clinical_patient,nte_er_positivity_other_scale,72,pos_finding_metastatic_breast_carcinoma_estrog...,receptor / biomarker,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]
6,clinical_patient,nte_er_positivity_define_method,73,metastatic_breast_carcinoma_estrogen_receptor_...,receptor / biomarker,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]
7,clinical_patient,nte_pr_positivity_other_scale,77,metastatic_breast_carcinoma_pos_finding_proges...,receptor / biomarker,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]
8,clinical_patient,nte_pr_positivity_define_method,78,metastatic_breast_carcinoma_progesterone_recep...,receptor / biomarker,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]
9,clinical_patient,nte_her2_positivity_other_scale,82,metastatic_breast_carcinoma_pos_finding_her2_e...,receptor / biomarker,1.0,0,0,False,True,False,True,weak_all_missing,low,[],[fill in during shortlist review]


In [9]:
unclear_manual_review_df = (
    shortlist_df.loc[shortlist_df['shortlist_bucket'] == 'unclear_manual_review', review_columns]
    .sort_values(
        ['manual_review_priority', 'support_likely_high_value', 'missing_like_fraction', 'table_name', 'source_position'],
        ascending=[True, False, True, True, True],
    )
    .reset_index(drop=True)
)
unclear_manual_review_path = results_root / '31_clinical_shortlist_unclear_manual_review_fields.tsv'
unclear_manual_review_df.to_csv(unclear_manual_review_path, sep='\t', index=False)

print(f'Saved: {unclear_manual_review_path}')
display(unclear_manual_review_df.head(25))


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\31_clinical_shortlist_unclear_manual_review_fields.tsv


,table_name,field_name,source_position,alternate_column_name,probable_field_group,missing_like_fraction,non_missing_count,distinct_non_missing_count,support_high_completeness,support_high_missingness,support_likely_high_value,support_likely_weak,shortlist_rule,manual_review_priority,example_values_small_sample,notes_placeholder
0,clinical_drug,bcr_patient_uuid,1,bcr_patient_uuid,identifier / admin,0.000000,2406,780,True,False,False,False,manual_identifier_admin,high,"[""C07B122E-AC50-4DB2-ADD2-5617A5D0E976"", ""9435...",[fill in during shortlist review]
1,clinical_drug,bcr_patient_barcode,2,bcr_patient_barcode,identifier / admin,0.000000,2406,780,True,False,False,False,manual_identifier_admin,high,"[""TCGA-GM-A2DA"", ""TCGA-AO-A03N"", ""TCGA-A2-A0EW...",[fill in during shortlist review]
2,clinical_drug,bcr_drug_barcode,3,bcr_drug_barcode,identifier / admin,0.000000,2406,2406,True,False,False,False,manual_identifier_admin,high,"[""TCGA-3C-AAAU-D60350"", ""TCGA-3C-AALI-D62900"",...",[fill in during shortlist review]
3,clinical_drug,bcr_drug_uuid,4,bcr_drug_uuid,identifier / admin,0.000000,2406,2406,True,False,False,False,manual_identifier_admin,high,"[""00300D73-B562-4EE9-A8D4-27E5B97AE501"", ""0051...",[fill in during shortlist review]
4,clinical_drug,form_completion_date,5,form_completion_date,identifier / admin,0.000000,2406,345,True,False,False,False,manual_identifier_admin,high,"[""2012-12-6"", ""2011-1-10"", ""2010-9-19"", ""2013-...",[fill in during shortlist review]
5,clinical_follow_up_v4_0,bcr_patient_uuid,1,bcr_patient_uuid,identifier / admin,0.000000,716,619,True,False,False,False,manual_identifier_admin,high,"[""23C31C2E-336C-4878-A476-CF8D811B4875"", ""28e9...",[fill in during shortlist review]
6,clinical_follow_up_v4_0,bcr_patient_barcode,2,bcr_patient_barcode,identifier / admin,0.000000,716,619,True,False,False,False,manual_identifier_admin,high,"[""TCGA-AR-A1AM"", ""TCGA-AR-A24Q"", ""TCGA-AR-A2LE...",[fill in during shortlist review]
7,clinical_follow_up_v4_0,bcr_followup_barcode,3,bcr_followup_barcode,identifier / admin,0.000000,716,716,True,False,False,False,manual_identifier_admin,high,"[""TCGA-3C-AAAU-F68069"", ""TCGA-3C-AAAU-F71341"",...",[fill in during shortlist review]
8,clinical_follow_up_v4_0,bcr_followup_uuid,4,bcr_followup_uuid,identifier / admin,0.000000,716,716,True,False,False,False,manual_identifier_admin,high,"[""0009B079-7981-4E93-BEDA-673A97FF1137"", ""0021...",[fill in during shortlist review]
9,clinical_follow_up_v4_0,form_completion_date,5,form_completion_date,identifier / admin,0.000000,716,210,True,False,False,False,manual_identifier_admin,high,"[""2013-10-2"", ""2013-3-2"", ""2015-3-30"", ""2014-1...",[fill in during shortlist review]
